<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/Assignment1_661_AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 1: Price Prediction using Neural Networks

## 1. Data Exploration and Cleaning

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")

In [ ]:
# import Airbnb Listing datasets for Asheville

listings_url = 'https://data.insideairbnb.com/united-states/nc/asheville/2024-06-21/data/listings.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')

Loading the dataset for data exploration and cleaning. Taken directly from W3.

In [ ]:
# Ex1 a: Display the first 10 rows
listings_df.head(10)

In [ ]:
# Ex1 b: Explore columns, data types, and non-null counts
listings_df.info()

In [ ]:
# Ex1 c: Display the shape of the DataFrame (rows, columns)
listings_df.shape

Initial data exploration. Taken directly from W3.

In [ ]:
# Ex 2: Print all unique host locations in the dataset
print(listings_df['host_location'].unique())      # List all location names
print(len(listings_df['host_location'].unique())) # Total number of unique locations

In [ ]:
# Ex 3: Filter Listings for Asheville
asheville_df = listings_df[listings_df['host_location'] == 'Asheville, NC']

In [ ]:
# Shape of the filtered dataset
asheville_df.shape

Ensure listings belong to hosts located in Asheville, NC. Kept all records with host_location as Asheville, dropped all rows with missing or different host_location values. Taken directly from W3.

In [ ]:
#Ex 4:  Define the columns we want to keep and create a new dataframe with the selected columns
selected_columns = [
    'price', 'bathrooms','bedrooms', 'number_of_reviews',
    'room_type', 'host_identity_verified', 'host_is_superhost',
    'accommodates', 'review_scores_rating'
]

asheville_df = asheville_df[selected_columns]

In [ ]:
# Preview the first few rows of the new dataframe
asheville_df.head()

Extended the selected features from W3 to include accommodates, review_scores_rating, and room_type. I removed latitude and longitute as I did not see them included in the instructions.

In [ ]:
# 5.1: Check for missing values in each column
asheville_df.isnull().sum()

Check for missing values before applying imputation strategies. Taken directly from W3.

In [ ]:
# 5.2: Drop rows where the target variable (price) is missing and then re-check for missing values
asheville_df = asheville_df.dropna(subset=['price'])
asheville_df.isnull().sum()

Drop rows where the price value is missing. Taken directly from W3.

In [ ]:
# 5.3: Use one of the imputation methods to impute missing values, if there are any missing values after dropping records in step 2
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].fillna(
    asheville_df['host_is_superhost'].mode()[0]
)
asheville_df.isnull().sum()

Impute host_is_superhost using mode. Taken directly from W3.

In [ ]:
# 5.4: Impute numerical missing values using mean or median, depending on data distribution
asheville_df['bedrooms'] = asheville_df['bedrooms'].fillna(asheville_df['bedrooms'].median())
asheville_df['bathrooms'] = asheville_df['bathrooms'].fillna(asheville_df['bathrooms'].mean())
asheville_df.isnull().sum()

Impute bedrooms using median and bathrooms using mean to fill missing values for numerical features except for review_scores_rating.

In [ ]:
# 5.5: For review_scores_rating, replace missing values with 0 (indicating no reviews)
asheville_df['review_scores_rating'] = asheville_df['review_scores_rating'].fillna(0)

# Create a dummy variable (has_review_scores) to indicate whether a listing has a review
asheville_df['has_review_scores'] = (asheville_df['review_scores_rating'] > 0).astype(int)

asheville_df.isnull().sum()

Replace missing values of review_scores_rating with 0. Then created a dummy variable has_review_scores to indicate whether a listing has a review. has_review_scores = 1 if review_scores_rating > 0, has_review_scores = 0 if review_scores_rating == 0 (no reviews).

In [ ]:
# check data types
asheville_df.dtypes

In [ ]:
# 6.2: Convert price to Numeric
asheville_df['price'] = asheville_df['price'].replace(r'[\$,]', '', regex=True).astype(float)

Convert price from string to numeric for analysis. Taken directly from W3.

In [ ]:
# 6.3: Convert Boolean-like Columns
asheville_df['host_identity_verified'] = asheville_df['host_identity_verified'].map({'t': 1, 'f': 0})
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].map({'t': 1, 'f': 0})

Dummy encode categorical variables host_identity_verified and host_is_superhost. Taken directly from W3.


In [ ]:
# 6.4: Encode Categorical Column

room_type_dummies = pd.get_dummies(asheville_df['room_type'], prefix='room_type', drop_first=True)
room_type_dummies = room_type_dummies.astype(int)

# Add new columns to the DataFrame
asheville_df = pd.concat([asheville_df, room_type_dummies], axis=1)
asheville_df.drop('room_type', axis=1, inplace=True)

asheville_df.head()

One-hot encode romm_type due to multi-category. Taken directly from W3.

In [ ]:
# Visualize the original price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (Before Removing Top 1%)")
plt.xlabel("Price")
plt.show()

In [ ]:
# Remove top 1% of extreme price values
price_cap = asheville_df['price'].quantile(0.99)
asheville_df = asheville_df[asheville_df['price'] <= price_cap]

In [ ]:
# Visualize the cleaned price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (After Removing Top 1%)")
plt.xlabel("Price")
plt.show()

Handle outliers. Taken directly from W3.

In [ ]:
# 7.1: Descriptive Statistics

asheville_df.describe()

Provide descriptive statistics. Taken directly from W3.

### Discussion and Reflection

The descriptive statistics provide useful insight into the distribution of Airbnb listings in Asheville. The average listing price is approximately $163, with a relatively large standard deviation of $112, indicating substantial variability in pricing even after removing extreme outliers. Most listings are modest in size, with a median of 1 bedroom and 1 bathroom, and they accommodate an average of about 4 guests. The high mean review score (4.63 out of 5) and the fact that 94% of listings have review scores suggest that most properties are well-rated and actively reviewed. Additionally, a large proportion of hosts are identity verified (89%) and superhosts (77%), indicating a generally established and reputable host base. The majority of listings are private rooms, while hotel and shared rooms represent a very small fraction of the dataset. Overall, the data exploration reveals meaningful variation in price, size, and host characteristics, which are likely to play an important role in predicting listing prices.

Additional preprocessing techniques that could improve model performance include: log-transforming the right-skewed price variable to reduce the influence of high-value listings, adding more predictive features from the original dataset (e.g., location, property type, amenities count), and applying feature scaling to variables with very different ranges before training.

## 2. Exploratory Data Analysis

In [ ]:
# 7.2: Histograms of Distributions

numeric_cols = ['price', 'bedrooms', 'bathrooms', 'number_of_reviews', 'accommodates', 'review_scores_rating']
asheville_df[numeric_cols].hist(bins=30, figsize=(12,10))
plt.tight_layout()
plt.show()

Updated the histogram code from W3 to include the new features accommodates and review_scores_rating to create an new histogram.

In [ ]:
corr = asheville_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

Calculate and visualize correlations between the variables. Taken directly from W3.

### Discussion

The histograms show that several variables are right-skewed, particularly price and number_of_reviews, indicating that most listings are moderately priced and have a relatively small number of reviews, while a smaller group has very high values. Bathrooms and bedrooms are concentrated around 1–2, suggesting that most listings are smaller properties. The review_scores_rating distribution is heavily concentrated near 5, indicating that most listings are highly rated, with very few low or zero ratings. The accommodates variable is centered around 2–6 guests, with fewer large-capacity properties.

The correlation matrix reveals that accommodates (0.66), bedrooms (0.63), and bathrooms (0.61) have the strongest positive relationships with price, suggesting that property size and capacity are key drivers of listing price. There is a strong correlation between bedrooms and accommodates (0.87) and between bedrooms and bathrooms (0.81), indicating potential multicollinearity among size-related features. Review-related variables show weak correlations with price, meaning ratings and number of reviews may have limited direct impact on price. Overall, the findings suggest that structural property characteristics are more influential for price prediction than review or host-related variables.

## 3. Model Building and Evaluation

In [ ]:
# Imports required libraries for pre-processing and modeling

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Normalization, Input
from sklearn.model_selection import train_test_split

# Selecting a subset of numeric features relevant to pricing.
features = ['bedrooms', 'bathrooms', 'number_of_reviews', 'host_is_superhost', 'host_identity_verified',
            'accommodates', 'review_scores_rating', 'has_review_scores',
            'room_type_Hotel room', 'room_type_Private room', 'room_type_Shared room']
target = 'price'

# Define X and y
X = asheville_df[features]
y = asheville_df[target]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define and Adapt Normalization Layer
norm_layer = Normalization()
norm_layer.adapt(X_train.values)

Updated the train test split code from W3 to include all assignment features and dummies from this assignment. Followed the same procedure as W3. 

In [ ]:
# Build the Sequential Model
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    norm_layer,
    Dense(64, activation='relu'),
    Dense(1)  # Regression output
])

Build a neural network regression model using Keras Sequential. Taken directly from W3.

In [ ]:
# Compile the Model
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

Updated the code from W3 to compile the model with optimizer with all the correct details for this assignment. Also made it also track mse.

In [ ]:
model.summary()

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=.2,
    epochs=50,
    verbose= 1
)

Train the model for 50 epochs with validation split of 20%. Taken directly from W3.

In [ ]:
import matplotlib.pyplot as plt

# Plot MAE over epochs
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('MAE (Price)')
plt.title('Training vs Validation MAE')
plt.legend()
plt.grid(True)
plt.show()

# Plot Loss (MSE) over epochs
plt.plot(history.history['loss'], label='Train Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

Training/validation loss curves. Taken directly from W3.

In [ ]:
# Evaluate on Test Set
test_loss, test_mae, test_mse = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MAE: ${test_mae:.2f}")
print(f"Test MSE: {test_mse:.2f}")
print(f"Test Loss: {test_loss:.2f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Predict on the test set
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"R² Score: {r2:.2f}")

Evaluate model performance using test set MSE, R². Taken directly from W3.

### Discussion

The model shows signs of **underfitting**. An R² of 0.46 means the model explains only 46% of the variance in listing prices, leaving over half unexplained. The training and validation loss curves converge closely together and flatten early, indicating the model is not complex enough to capture the underlying patterns rather than memorizing the training data (which would indicate overfitting).

**Suggested improvements:**
- Add more hidden layers or increase the number of nodes to give the model more capacity to learn complex relationships.
- Increase the number of training epochs to allow the model more time to converge.
- Include additional features from the dataset (e.g., location, amenities, property type) that may be strong predictors of price.
- Apply feature engineering such as log-transforming the skewed price variable to help the model learn more effectively.